# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [6]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/Users/abhishekkarthikakunuru/Documents/data-viz-class-material-w2026/data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [7]:
# Task 1
df_fossil = df.loc[df['Source_Type'] == 'Fossil']

# find the biggest fossil user for the title
top_region = df_fossil.groupby('Region')['TWh'].sum().idxmax()
top_country = df_fossil.groupby('Country')['TWh'].sum().idxmax()
top_twh = df_fossil.groupby('Country')['TWh'].sum().max()

# Okabe-Ito colours (CVD safe), '(?)' is the mixed parent node so it goes grey
colors = {'Coal': '#D55E00', 'Oil': '#0072B2', 'Natural Gas': '#E69F00', '(?)': 'lightgrey'}

fig1 = px.treemap(df_fossil, path=['Region', 'Country', 'Source'], values='TWh',
                  color='Source', color_discrete_map=colors,
                  title=f'{top_region} uses the most fossil energy, {top_country} alone burns {top_twh:,.0f} TWh')
fig1.update_traces(texttemplate='%{label}<br>%{value:,.0f} TWh', hovertemplate='%{label}: %{value:,.0f} TWh')
fig1.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [8]:
# Task 2 - charts
df_low = df.loc[df['Source_Type'] == 'Low-carbon'].groupby('Country', as_index=False)['TWh'].sum()
df_low['All'] = 'Low-carbon'
df_low = df_low.sort_values('TWh')

# treemap
fig2a = px.treemap(df_low, path=['All', 'Country'], values='TWh', title='Low-carbon energy (Nuclear + Hydro) by country')
fig2a.update_traces(texttemplate='%{label}<br>%{value:,.0f} TWh')
fig2a.show()

# bar chart, sorted ascending so the biggest ends up on top
top = df_low.iloc[-1]
fig2b = px.bar(df_low, x='TWh', y='Country', orientation='h', text='TWh',
               color_discrete_sequence=['#0072B2'],
               title=f"{top['Country']} leads low-carbon energy with {top['TWh']:,.0f} TWh")
fig2b.update_traces(texttemplate='%{text:,.0f} TWh', textposition='outside')
fig2b.show()


### Task 2 - reflection

**Which chart tells the ranking story more clearly, and why?**

The bar chart. All bars start from the same baseline and are sorted, so I can read the ranking straight away and see the gaps between countries. In the treemap I have to compare rectangle areas with different shapes, which is hard, especially for the smaller countries that are close in size.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [9]:
# Task 3
df_reg = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
df_reg['Share'] = df_reg['TWh'] / df_reg.groupby('Region')['TWh'].transform('sum') * 100

# order regions by coal share, highest first
coal = df_reg.loc[df_reg['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist()

# label only the coal segments
df_reg['Label'] = np.where(df_reg['Source'] == 'Coal', df_reg['Share'].round().astype(int).astype(str) + '%', '')

fig3 = px.bar(df_reg, x='Share', y='Region', color='Source', orientation='h', text='Label',
              category_orders={'Source': ['Coal', 'Oil', 'Natural Gas'], 'Region': region_order},
              color_discrete_map={'Coal': '#D55E00', 'Oil': '#BBBBBB', 'Natural Gas': '#888888'},
              title=f"{coal.iloc[0]['Region']} relies most on coal, {coal.iloc[0]['Share']:.0f}% of its fossil energy")
fig3.update_layout(barmode='stack', xaxis_title='Share of fossil TWh (%)')
fig3.show()


### Task 3 - reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

Treemap: which region or country uses the most fossil energy in absolute TWh (for example how big Asia's coal block is compared to everything else). The stacked bar can't show this because every bar is scaled to 100%.

Stacked bar: which region depends most on coal as a share of its fossil mix. The treemap can't answer this well because the regions have very different sizes, so comparing proportions inside them by eye doesn't work.